# En elv i luften
## Ekstremværet Jakob og den atmosfæriske elven, oktober 2024

Torsdag 31. oktober 2024 traff en atmosfærisk elv Vestlandet. Meteorologisk institutt sendte ut rødt farevarsel for ekstremt mye regn i midtre og indre strøk av Vestland, og ekstremværet fikk navnet Jakob. Gullfjellet i Bergen fikk 206 mm regn på 24 timer. Ni stasjoner fikk mer nedbør enn det som i gjennomsnitt kommer en gang hvert 50. år. Bruer ble tatt, veier ble stengt, og Bergensbanen sto.

I denne økten bruker du ekte data til å svare på hovedspørsmålet nedenfor.

> 🌧️ **Hovedspørsmål**
> Hvordan kan luft over Atlanterhavet gi over 200 mm regn på ett døgn på Vestlandet, og hva skjer med slike hendelser i et varmere klima?

| Tid | Del |
|---|---|
| 0–10 min | Oppstart og hypotese |
| 10–35 min | Del 1 Elven i luften |
| 35–60 min | Del 2 Når elven treffer fjellene |
| 60–70 min | Pause |
| 70–100 min | Del 3 Varmere luft, mer regn? |
| 100–120 min | Del 4 Oppsummering |

**Slik jobber du**
- Kjør cellene i rekkefølge med Shift + Enter.
- Svar i cellene merket ✏️. Dobbeltklikk for å skrive, og trykk Shift + Enter når du er ferdig.
- I cellene merket 🔧 skal du endre en verdi selv og kjøre cellen på nytt.
- Har du tid til overs, finner du to fordypningsoppgaver helt til slutt.

### Oppstart
Hvor kommer vannet fra når det regner 200 mm på ett døgn? Skriv en hypotese. Tenk på hvor mye vann luften rett over deg inneholder, og hva som må til for at så mye vann skal falle på samme sted.

✏️ **Svar**

*Min hypotese er at …*

### Last inn verktøy og data
Kjør cellen nedenfor. Den henter inn Python-pakkene og leser datafilene i mappen `data`. Les meldingene som skrives ut.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
import xarray as xr
import cartopy.crs as ccrs
import cartopy.feature as cfeature

DATA = Path("data")
PC = ccrs.PlateCarree()           # koordinatene i dataene er lengde- og breddegrad
PROJ = ccrs.LambertConformal(central_longitude=-10, standard_parallels=(45, 65))
ATLANTEREN = [-50, 25, 35, 72]    # V, Ø, S, N
VESTLANDET = [3, 13, 57.5, 63.5]
KM_PER_GRAD = 111.2               # km per breddegrad

plt.rcParams.update({"figure.dpi": 110, "font.size": 10,
                     "axes.grid": True, "grid.alpha": 0.3})

STEDER = {"Bergen": (60.39, 5.33), "Voss": (60.63, 6.42), "Odda": (60.07, 6.55),
          "Kinsarvik": (60.38, 6.72), "Førde": (61.45, 5.85)}
FARGER = ["#0072B2", "#D55E00", "#009E73", "#E69F00", "#CC79A7", "#000000"]


def komma(ax, x=False, y=True):
    """Desimalkomma og ekte minus på aksene."""
    f = mticker.FuncFormatter(lambda v, p: f"{v:g}".replace(".", ",").replace("-", "−"))
    if x:
        ax.xaxis.set_major_formatter(f)
    if y:
        ax.yaxis.set_major_formatter(f)


def tidsakse(ax):
    ax.xaxis.set_major_locator(mdates.HourLocator(byhour=[0, 12]))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d.%m\nkl. %H"))


def kart(ax, utsnitt=ATLANTEREN, land=True):
    """Kartbakgrunn med kystlinje og gradnett med norske retninger."""
    ax.set_extent(utsnitt, crs=PC)
    if land:
        ax.add_feature(cfeature.LAND, facecolor="0.92", zorder=0)
    ax.coastlines(resolution="50m", lw=0.6, color="0.2", zorder=3)
    gl = ax.gridlines(draw_labels=True, x_inline=False, y_inline=False,
                      lw=0.4, color="0.5", ls=":")
    gl.top_labels = gl.right_labels = False
    gl.xformatter = mticker.FuncFormatter(
        lambda v, p=None: "0°" if v == 0 else f"{abs(v):.0f}° {'V' if v < 0 else 'Ø'}")
    gl.yformatter = mticker.FuncFormatter(lambda v, p=None: f"{v:.0f}° N")


def merk_steder(ax, steder=STEDER, farge="k"):
    for navn, (la, lo) in steder.items():
        ax.plot(lo, la, "o", ms=3, color=farge, transform=PC, zorder=6)
        ax.text(lo + 0.15, la + 0.08, navn, fontsize=7, color=farge, transform=PC, zorder=6)


def tall(v, des=1):
    return f"{v:,.{des}f}".replace(",", " ").replace(".", ",").replace("-", "−")


def finnes(fil):
    if (DATA / fil).exists():
        return True
    print(f"⚠️  Finner ikke {fil} i mappen data. Spør læreren.")
    return False


kartdata = xr.open_dataset(DATA / "era5_jakob_kart.nc") if finnes("era5_jakob_kart.nc") else None
nedbor = xr.open_dataset(DATA / "era5_jakob_nedbor.nc") if finnes("era5_jakob_nedbor.nc") else None
if kartdata is not None:
    kartdata["ivt"] = np.hypot(kartdata.ivt_u, kartdata.ivt_v)

PENE_NAVN = {"ovstedal": "Øvstedal", "gullfjellet": "Gullfjellet", "haukedal": "Haukedal"}
stasjoner = {}
for f in sorted(DATA.glob("stasjon_*_time.csv")):
    kort = f.stem.replace("stasjon_", "").replace("_time", "")
    s = pd.read_csv(f).set_index("tid").nedbor
    s.index = pd.to_datetime(s.index, utc=True).tz_localize(None)    # tid i UTC
    stasjoner[PENE_NAVN.get(kort, kort.capitalize())] = s
print("Stasjoner med timesnedbør:", ", ".join(stasjoner) or "ingen")
print("Klar ✅")

---
## Del 1 Elven i luften (25 min)

> 📘 **Begrep** **Vanndamptransport** (IVT, fra engelsk *integrated vapour transport*) er hvor mye vanndamp som strømmer gjennom en tenkt vegg som er 1 meter bred og går fra bakken til toppen av atmosfæren. Enheten er kg per meter per sekund, kg/(m·s). IVT blir stor når luften både er fuktig og blåser kraftig.

> 📘 **Begrep** En **atmosfærisk elv** er et langt og smalt belte med stor vanndamptransport. En vanlig tommelfingerregel er at IVT er minst 250 kg/(m·s) i et belte som er over 2000 km langt og under 1000 km bredt. Elvene ligger vanligvis langs kaldfronten i et lavtrykk, i den varme luften foran fronten.

### 1a Elven på kartet
Fargene viser IVT. Pilene viser retningen vanndampen strømmer mot. Svarte linjer er isobarer hver 5. hPa. Den tykke linjen markerer 250 kg/(m·s). Dataene kommer fra ERA5.

> 📘 **Begrep** **ERA5** er en **reanalyse**. Det betyr at en værvarslingsmodell har regnet seg fram til været time for time, styrt av alle målinger som finnes. Resultatet er et komplett datasett for hele jorden fra 1940 til i dag, med ruter på omtrent 25 × 25 km.

In [ ]:
TIDER = ["2024-10-30T12", "2024-10-31T06", "2024-11-01T00"]     # 🔧

if kartdata is not None:
    fig, akser = plt.subplots(1, 3, figsize=(15, 5.2), subplot_kw={"projection": PROJ},
                              constrained_layout=True)
    for ax, t in zip(akser, TIDER):
        k = kartdata.sel(time=t, method="nearest")
        kart(ax, land=False)
        cf = ax.contourf(k.lon, k.lat, k.ivt, levels=np.arange(0, 1101, 100),
                         cmap="YlGnBu", extend="max", transform=PC)
        ax.contour(k.lon, k.lat, k.ivt, levels=[250], colors="k", linewidths=1.8,
                   transform=PC)
        cs = ax.contour(k.lon, k.lat, k.msl, levels=np.arange(940, 1050, 5), colors="0.25",
                        linewidths=0.6, transform=PC)
        ax.clabel(cs, fmt="%d", fontsize=6)
        n = 6
        ax.quiver(k.lon.values[::n], k.lat.values[::n],
                  k.ivt_u.values[::n, ::n], k.ivt_v.values[::n, ::n],
                  transform=PC, scale=15000, width=0.003, color="#D55E00", zorder=4)
        merk_steder(ax, {"Bergen": STEDER["Bergen"]})
        ax.set_title(f"{pd.Timestamp(k.time.values):%d.%m.%Y kl. %H} UTC")
    fig.colorbar(cf, ax=akser, shrink=0.75, label="Vanndamptransport, IVT (kg/(m·s))")
    fig.text(0.01, 0.0, "Kilde: ERA5, Copernicus/ECMWF. Piler viser retningen vanndampen "
             "strømmer mot. Lambert konform kjegleprojeksjon.", fontsize=7)
    plt.show()

    for t in TIDER:
        kyst = kartdata.ivt.sel(time=t, method="nearest").sel(lat=slice(58, 63), lon=slice(2, 6))
        print(f"{pd.Timestamp(t):%d.%m kl. %H} UTC  største IVT utenfor Vestlandet: "
              f"{tall(float(kyst.max()), 0)} kg/(m·s)")

**Oppgaver**
1. Beskriv formen på området med IVT over 250 kg/(m·s). Bruk gradnettet til å anslå hvor lang og bred elven er (1 breddegrad er omtrent 111 km).
2. Hvor kommer vanndampen fra? Følg pilene bakover.
3. Hvor ligger elven i forhold til lavtrykket? Se på isobarene.
4. Hvor lenge lå elven over Vestlandet?

✏️ **Svar**

*Skriv svaret ditt her.*

### 1b Vanndamp og vanndamptransport
**Vanndampinnhold** (total kolonnevanndamp) er hvor mye vanndamp det er i en søyle av luft fra bakken til toppen av atmosfæren. Det oppgis i kg/m², som er det samme som millimeter vann hvis all vanndampen kondenserte.

In [ ]:
TID_B = "2024-10-31T06"     # 🔧

if kartdata is not None:
    k = kartdata.sel(time=TID_B, method="nearest")
    fig = plt.figure(figsize=(8, 6.2))
    ax = plt.axes(projection=PROJ)
    kart(ax, land=False)
    cf = ax.contourf(k.lon, k.lat, k.tcwv, levels=np.arange(0, 46, 5), cmap="BuPu",
                     extend="max", transform=PC)
    ax.contour(k.lon, k.lat, k.ivt, levels=[250], colors="k", linewidths=1.8, transform=PC)
    merk_steder(ax, {"Bergen": STEDER["Bergen"]})
    fig.colorbar(cf, ax=ax, shrink=0.75, label="Vanndampinnhold (mm)")
    ax.set_title(f"Vanndampinnhold {pd.Timestamp(k.time.values):%d.%m.%Y kl. %H} UTC. "
                 "Svart linje er IVT = 250 kg/(m·s)", fontsize=9)
    fig.text(0.02, 0.02, "Kilde: ERA5, Copernicus/ECMWF", fontsize=7)
    plt.show()
    print(f"Vanndampinnhold over Bergen: {tall(float(k.tcwv.sel(lat=60.4, lon=5.3, method='nearest')))} mm")

**Oppgaver**
1. Hvor mye vanndamp var det i luften over Bergen? Sammenlign med de 206 mm som kom på Gullfjellet. Hva forteller det deg om hvor vannet kom fra?
2. Det fuktigste området og området med størst IVT er ikke helt det samme. Forklar hvorfor. Tenk på hva IVT består av.

✏️ **Svar**

*Skriv svaret ditt her.*

### 1c Hvor stor er elven?
Vi kan regne ut hvor mye vanndamp som strømmer gjennom et snitt på tvers av elven. Snittet går langs en lengdegrad. Da er det den østlige komponenten av IVT som går gjennom snittet.

In [ ]:
LON_SNITT = -5            # 🔧 lengdegrad for snittet
TID_C = "2024-10-31T06"   # 🔧
LAT_FRA, LAT_TIL = 50, 70

if kartdata is not None:
    snitt = kartdata.ivt_u.sel(time=TID_C, method="nearest").sel(
        lon=LON_SNITT, method="nearest").sel(lat=slice(LAT_FRA, LAT_TIL))
    dy = float(abs(snitt.lat[1] - snitt.lat[0])) * KM_PER_GRAD * 1000     # meter
    elv = snitt.where(snitt >= 250)
    transport = float(elv.sum()) * dy                                     # kg/s
    bredde = float(elv.notnull().sum()) * dy / 1000                       # km

    fig, ax = plt.subplots(figsize=(8, 3.8), constrained_layout=True)
    ax.plot(snitt.lat, snitt, color="#0072B2", lw=1.8)
    ax.fill_between(snitt.lat, 250, snitt, where=snitt >= 250, color="#56B4E9", alpha=0.5,
                    hatch="//", label="Den atmosfæriske elven")
    ax.axhline(250, color="k", ls="--", lw=1)
    ax.set_xlabel("Breddegrad (° N)")
    ax.set_ylabel("IVT østover (kg/(m·s))")
    ax.set_title(f"Snitt langs {abs(LON_SNITT)}° {'V' if LON_SNITT < 0 else 'Ø'}, "
                 f"{pd.Timestamp(snitt.time.values):%d.%m kl. %H} UTC")
    ax.legend()
    komma(ax, x=True)
    plt.show()

    print(f"Bredde på elven: omtrent {tall(bredde, 0)} km")
    print(f"Vanndamptransport gjennom snittet: {tall(transport / 1000, 0)} tonn per sekund")
    print(f"Som flytende vann: {tall(transport / 1000, 0)} m³/s (1 tonn vann er 1 m³)")
    print("Til sammenligning: Glomma ca. 700 m³/s, Amazonas ca. 200 000 m³/s (middelvannføring)")

**Oppgaver**
1. Hvor mye vann fraktet elven gjennom snittet hvert sekund? Hvor mange Glommaer tilsvarer det?
2. Flytt snittet til en annen lengdegrad eller et annet tidspunkt. Hvordan endrer transporten seg?
3. Hvorfor kaller vi fenomenet en elv?

✏️ **Svar**

*Skriv svaret ditt her.*

---
## Del 2 Når elven treffer fjellene (25 min)

### 2a Hvor kom regnet?
Kartet viser ERA5-nedbøren i perioden du velger. Grå linjer viser terrenghøyden i ERA5 for hver 500. meter.

In [ ]:
FRA, TIL = "2024-10-30T00", "2024-11-01T06"     # 🔧

if nedbor is not None:
    sum_n = nedbor.nedbor.sel(time=slice(FRA, TIL)).sum("time")
    fig = plt.figure(figsize=(8, 6.5))
    ax = plt.axes(projection=ccrs.LambertConformal(central_longitude=8,
                                                   standard_parallels=(58, 63)))
    kart(ax, VESTLANDET, land=False)
    cf = ax.contourf(sum_n.lon, sum_n.lat, sum_n, levels=np.arange(0, 221, 20),
                     cmap="YlGnBu", extend="max", transform=PC)
    cs = ax.contour(nedbor.lon, nedbor.lat, nedbor.hoyde, levels=[500, 1000, 1500],
                    colors="0.35", linewidths=[0.5, 0.8, 1.1], transform=PC)
    ax.clabel(cs, fmt="%d m", fontsize=6)
    merk_steder(ax)
    fig.colorbar(cf, ax=ax, shrink=0.75, label="Nedbør (mm)")
    ax.set_title(f"Nedbør i ERA5 fra {FRA[8:10]}.{FRA[5:7]} kl. {FRA[11:]} "
                 f"til {TIL[8:10]}.{TIL[5:7]} kl. {TIL[11:]} UTC")
    fig.text(0.02, 0.02, "Kilde: ERA5, Copernicus/ECMWF. Grå linjer viser terrenghøyde i ERA5.",
             fontsize=7)
    plt.show()
    print(f"Mest nedbør i en ERA5-rute: {tall(float(sum_n.max()))} mm")

**Oppgaver**
1. Hvor kom det mest nedbør? Ligger området ved kysten, i fjordene eller på fjellet?
2. Hvor mye nedbør kom øst for vannskillet, for eksempel over Hardangervidda?
3. Den høyeste stasjonsverdien var 206 mm. Hvorfor viser ERA5 mindre?

✏️ **Svar**

*Skriv svaret ditt her.*

### 2b Snitt fra kysten og innover

In [ ]:
LAT_PROFIL = 60.6     # 🔧 breddegrad for profilen (Voss ligger på 60,6° N)

if nedbor is not None:
    p = sum_n.sel(lat=LAT_PROFIL, method="nearest")
    h = nedbor.hoyde.sel(lat=LAT_PROFIL, method="nearest")
    fig, a1 = plt.subplots(figsize=(9, 3.8), constrained_layout=True)
    a1.fill_between(h.lon, 0, h, color="0.6", label="Terrenghøyde i ERA5")
    a1.set_ylabel("Terrenghøyde (m)")
    a1.set_xlabel("Lengdegrad (° Ø)")
    a2 = a1.twinx()
    a2.plot(p.lon, p, color="#0072B2", marker="o", ms=3, lw=1.8, label="Nedbør")
    a2.set_ylabel("Nedbør (mm)")
    a2.grid(False)
    a1.set_title(f"Fra kysten og innover langs {tall(LAT_PROFIL)}° N")
    a1.legend(loc="upper left"); a2.legend(loc="upper right")
    komma(a1, x=True); komma(a2)
    plt.show()

> 📘 **Begrep** **Orografisk nedbør** oppstår når luft blir tvunget opp over fjell. Luften blir avkjølt når den stiger, vanndampen kondenserer, og det dannes skyer og nedbør. På lesiden av fjellene synker luften og blir varmet opp. Der blir det **regnskygge**.

**Oppgaver**
1. Beskriv hvordan nedbøren endrer seg fra kysten og innover mot fjellene. Hvor er nedbøren størst?
2. Forklar mønsteret med orografisk nedbør og regnskygge.
3. MET skrev at sterk vind i høyden presset regnet langt inn i fjordene og fjellene. Hvorfor kom det da mye nedbør også i indre strøk denne gangen?

✏️ **Svar**

*Skriv svaret ditt her.*

### 2c Et enkelt regnestykke
Hvor mye regn kan en atmosfærisk elv gi? Anta at vanndampen som strømmer inn over kysten, regner ut over en strekning med lengde L. Bare en andel e av vanndampen blir til nedbør, resten fortsetter videre. Da blir nedbørintensiteten omtrent

$$ P \approx \frac{e \cdot \text{IVT}}{L} $$

med P i kg/(m²·s). Gang med 3600 for å få mm per time.

In [ ]:
IVT_KYST = 600      # 🔧 kg/(m·s), bruk tallet du fant i 1a
ANDEL = 0.4         # 🔧 andel av vanndampen som regner ut
L_KM = 150          # 🔧 strekning fra kysten til vannskillet (km)
TIMER = 24          # 🔧 hvor lenge elven ligger i ro

P = ANDEL * IVT_KYST / (L_KM * 1000)                # kg/(m²·s)
print(f"Nedbørintensitet {tall(P * 3600)} mm per time")
print(f"Nedbør på {TIMER} timer: {tall(P * 3600 * TIMER, 0)} mm")

**Oppgaver**
1. Bruk IVT-verdien du fant utenfor Vestlandet i 1a. Hvor mye nedbør gir regnestykket på 24 timer? Sammenlign med 206 mm på Gullfjellet.
2. Hvilken av størrelsene i regnestykket er mest usikker? Hvordan påvirker fjellene andelen e?

✏️ **Svar**

*Skriv svaret ditt her.*

### 2d Regnet time for time
Hvis læreren har lastet ned stasjonsdata, ser du målinger fra stasjoner MET nevner i rapporten om Jakob. Ellers ser du ERA5 for Bergen.

In [ ]:
fig, (a1, a2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True, constrained_layout=True)
serier = dict(stasjoner)
if not serier and nedbor is not None:
    serier = {"Bergen (ERA5)": nedbor.nedbor.sel(lat=60.4, lon=5.3, method="nearest").to_series()}
bredde = 0.8 / 24 / max(len(serier), 1)
for i, (navn, s) in enumerate(serier.items()):
    s = s.loc["2024-10-30":"2024-11-01"]
    a1.bar(s.index + pd.Timedelta(hours=i * 0.8 / len(serier)), s, width=bredde,
           color=FARGER[i], label=navn)
    a2.plot(s.index, s.cumsum(), color=FARGER[i], lw=1.8,
            ls=["-", "--", ":", "-."][i % 4], label=navn)
    print(f"{navn:12s} sum {tall(s.sum())} mm, mest på én time {tall(s.max())} mm "
          f"({s.idxmax():%d.%m kl. %H} UTC)")
a1.set_ylabel("Nedbør per time (mm)")
a1.legend(fontsize=8)
a2.set_ylabel("Akkumulert nedbør (mm)")
a2.legend(fontsize=8)
tidsakse(a2); komma(a1); komma(a2)
a1.set_title("Nedbør time for time")
plt.show()

**Oppgaver**
1. Når regnet det mest? Hvor lenge varte det kraftigste regnet?
2. Hvor mye regn kom på det meste i løpet av én time? Var det et kort skybrudd eller jevnt og langvarig regn?

✏️ **Svar**

*Skriv svaret ditt her.*

### 2e Nullgradersgrensen
Hvor høyt over havet temperaturen er 0 °C avgjør om nedbøren kommer som regn eller snø i fjellet.

In [ ]:
STED = "Voss"     # 🔧

if kartdata is not None and nedbor is not None:
    la, lo = STEDER[STED]
    null = kartdata.nullgrad.sel(lat=la, lon=lo, method="nearest").to_series()
    n = nedbor.nedbor.sel(lat=la, lon=lo, method="nearest").to_series()
    fig, a1 = plt.subplots(figsize=(10, 3.8), constrained_layout=True)
    a1.bar(n.index, n, width=1 / 24, color="#56B4E9", label="Nedbør per time (ERA5)")
    a1.set_ylabel("Nedbør per time (mm)")
    a2 = a1.twinx()
    a2.plot(null.index, null, color="#D55E00", marker="o", ms=3, label="Nullgradersgrensen")
    a2.set_ylabel("Høyde (m over havet)")
    a2.grid(False)
    a1.legend(loc="upper left"); a2.legend(loc="upper right")
    a1.set_title(f"Nedbør og nullgradersgrensen ved {STED}")
    tidsakse(a1); komma(a1); komma(a2)
    plt.show()

**Oppgaver**
1. Hvor høyt lå nullgradersgrensen mens det regnet mest? Hva betyr det for nedbøren i fjellet?
2. Hva skjedde med nullgradersgrensen etter at elven hadde passert? Forklar hvorfor.
3. Hvorfor gir regn langt opp i fjellet større flomfare enn snø?

✏️ **Svar**

*Skriv svaret ditt her.*

---
## ☕ Pause (10 min)

---
## Del 3 Varmere luft, mer regn? (30 min)

### 3a Hvor mye vanndamp kan luften holde?
Hvor mye vanndamp luften kan inneholde, øker raskt med temperaturen. Sammenhengen kalles **Clausius–Clapeyron-ligningen**. Her bruker vi en tilnærming (Magnus-formelen) for metningstrykket til vanndamp

$$ e_s(T) = 6{,}112 \cdot \exp\left(\frac{17{,}62\, T}{243{,}12 + T}\right) $$

der $e_s$ er i hPa og $T$ i °C.

In [ ]:
T = np.linspace(-10, 30, 200)
es = 6.112 * np.exp(17.62 * T / (243.12 + T))

T0 = 10           # 🔧 temperatur i °C
es0 = 6.112 * np.exp(17.62 * T0 / (243.12 + T0))
es1 = 6.112 * np.exp(17.62 * (T0 + 1) / (243.12 + T0 + 1))
okning = (es1 / es0 - 1) * 100

fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
ax.plot(T, es, color="#0072B2", lw=2)
ax.plot([T0, T0 + 1], [es0, es1], "o", color="#D55E00")
ax.set_xlabel("Temperatur (°C)")
ax.set_ylabel("Metningstrykk for vanndamp (hPa)")
ax.set_title("Hvor mye vanndamp luften kan inneholde")
komma(ax, x=True)
plt.show()
print(f"Ved {T0} °C øker metningstrykket med {tall(okning)} % per grad")

**Oppgaver**
1. Hvor mange prosent mer vanndamp kan luften inneholde per grad oppvarming? Prøv flere temperaturer.
2. Anta at luften i elven hadde vært 2 °C varmere, og at nedbøren øker like mye som vanndampen. Hvor mye regn ville kommet på Gullfjellet i stedet for 206 mm?
3. Nevn en grunn til at nedbøren i virkeligheten kan øke mer eller mindre enn dette.

✏️ **Svar**

*Skriv svaret ditt her.*

### 3b Har det blitt mer kraftig nedbør i Bergen?
Du kan velge mellom to dataserier for Bergen. Stasjonen Bergen – Florida har målinger fra MET, men serien starter først i 1983. ERA5-serien går tilbake til 1940, men er en modellrute på omtrent 25 × 25 km som jevner ut de største regnmengdene.

In [ ]:
KILDE = "stasjon"   # 🔧 "stasjon" (Bergen – Florida fra 1983) eller "era5" (Bergen fra 1940)
TERSKEL = 50        # 🔧 mm på ett døgn

if KILDE == "stasjon" and (DATA / "stasjon_florida_dogn.csv").exists():
    b = pd.read_csv(DATA / "stasjon_florida_dogn.csv", parse_dates=["dato"]).set_index("dato").nedbor
    kilde = "Bergen – Florida (MET Norge)"
else:
    if KILDE == "stasjon":
        print("Finner ikke stasjonsdata, bruker ERA5 i stedet.")
    b = pd.read_csv(DATA / "punkt_bergen_nedbor.csv", parse_dates=["dato"]).set_index("dato").nedbor
    kilde = "Bergen (ERA5 via Open-Meteo)"
b = b[b.index.year <= 2025]           # bare hele år

aar = b.groupby(b.index.year)
rx1 = aar.max()
dager = (b >= TERSKEL).groupby(b.index.year).sum()
total = aar.sum()

fig, akser = plt.subplots(3, 1, figsize=(10, 8), sharex=True, constrained_layout=True)
akser[0].bar(total.index, total, color="#56B4E9", edgecolor="k", lw=0.2)
akser[0].plot(total.index, total.rolling(30, center=True, min_periods=20).mean(), color="k", lw=2)
akser[0].set_ylabel("Årsnedbør (mm)")
akser[1].plot(rx1.index, rx1, "o", color="#0072B2", ms=3)
akser[1].plot(rx1.index, rx1.rolling(30, center=True, min_periods=20).mean(), color="k", lw=2)
akser[1].set_ylabel("Største døgnnedbør (mm)")
akser[2].bar(dager.index, dager, color="#D55E00", edgecolor="k", lw=0.2)
akser[2].set_ylabel(f"Døgn med minst {TERSKEL} mm")
akser[2].set_xlabel("År")
akser[0].set_title(f"{kilde}, {total.index.min()}–{total.index.max()}. "
                   "Svart linje er glidende 30-årsmiddel")
for a in akser:
    komma(a)
plt.show()

print(f"Data fra {total.index.min()} til {total.index.max()}")
for navn, (a, c) in {"1961–1990": (1961, 1990), "1991–2020": (1991, 2020)}.items():
    n = total.loc[a:c].size
    if n == 0:
        print(f"{navn}  ingen data i denne perioden")
        continue
    merknad = "" if n == 30 else f"  ⚠️ bare {n} av 30 år"
    print(f"{navn}  årsnedbør {tall(total.loc[a:c].mean(), 0)} mm, "
          f"største døgnnedbør {tall(rx1.loc[a:c].mean())} mm, "
          f"døgn over {TERSKEL} mm {tall(dager.loc[a:c].mean())} per år{merknad}")

**Oppgaver**
1. Hvordan har årsnedbøren og den største døgnnedbøren i Bergen endret seg?
2. Har antallet døgn med kraftig nedbør endret seg? Prøv en annen terskel.
3. Bytt til `KILDE = "era5"` og kjør cellen på nytt. Hva skjer med sammenligningen mellom 1961–1990 og 1991–2020? Hvorfor gir ERA5 lavere verdier for den største døgnnedbøren enn stasjonen?
4. Hvilken av de to seriene ville du brukt for å si noe om endring over tid, og hvorfor?
5. Forskere ved Bjerknessenteret har funnet at nedbøren i Norge har økt med om lag 20 % siden 1900, mest på Sørvestlandet. Passer resultatene dine med dette?

✏️ **Svar**

*Skriv svaret ditt her.*

---
## Del 4 Oppsummering (20 min)

Gå tilbake til hovedspørsmålet og hypotesen din fra starten.

> 🌧️ Hvordan kan luft over Atlanterhavet gi over 200 mm regn på ett døgn på Vestlandet, og hva skjer med slike hendelser i et varmere klima?

Skriv et svar på 150–250 ord. Bruk minst **tre konkrete tall** fra figurene og utregningene ovenfor, og vis til hvor tallene kommer fra. Forklar hvorfor **både** den atmosfæriske elven **og** fjellene trengs for å få så mye regn. Skriv også om hypotesen din stemte.

✏️ **Svar**

*Den atmosfæriske elven …*

**Refleksjon**
MET sendte ut rødt farevarsel for 120–180 mm på 24 timer, og stasjonene målte opptil 206 mm. Var varselet godt? Hva gjør det vanskelig å varsle nøyaktig hvor mye regn som kommer på ett bestemt sted?

✏️ **Svar**

*Skriv svaret ditt her.*

---
## Fordypning A Elven gjennom fire døgn

In [ ]:
if kartdata is not None:
    tider = pd.date_range("2024-10-29T00", "2024-11-01T12", freq="12h")
    fig, akser = plt.subplots(2, 4, figsize=(15, 7.5), subplot_kw={"projection": PROJ},
                              constrained_layout=True)
    for ax, t in zip(akser.flat, tider):
        k = kartdata.sel(time=t, method="nearest")
        kart(ax, land=False)
        cf = ax.contourf(k.lon, k.lat, k.ivt, levels=np.arange(0, 1101, 100),
                         cmap="YlGnBu", extend="max", transform=PC)
        ax.contour(k.lon, k.lat, k.ivt, levels=[250], colors="k", linewidths=1.2, transform=PC)
        ax.set_title(f"{t:%d.%m kl. %H} UTC", fontsize=9)
    fig.colorbar(cf, ax=akser, shrink=0.6, label="IVT (kg/(m·s))")
    plt.show()

**Oppgaver**
1. Beskriv hvordan elven oppstår, flytter seg og forsvinner.
2. Hvor lenge lå elven over omtrent samme sted på Vestlandet? Hvorfor er varigheten så viktig for hvor mye regn det blir?

✏️ **Svar**

*Skriv svaret ditt her.*

## Fordypning B Lag din egen elvedetektor
Cellen teller hvor stort område som har IVT over terskelen nær Norge til hver tid.

In [ ]:
TERSKEL_IVT = 250       # 🔧 kg/(m·s)

if kartdata is not None:
    omr = kartdata.ivt.sel(lat=slice(50, 68), lon=slice(-30, 15))
    celle = (0.5 * KM_PER_GRAD) ** 2 * np.cos(np.radians(omr.lat))     # km² per rute
    areal = ((omr >= TERSKEL_IVT) * celle).sum(["lat", "lon"]) / 1e6        # mill. km²
    fig, ax = plt.subplots(figsize=(10, 3.5), constrained_layout=True)
    ax.plot(areal.time, areal, color="#0072B2", marker="o", ms=3)
    ax.set_ylabel("Areal (millioner km²)")
    ax.set_title(f"Areal med IVT over {TERSKEL_IVT} kg/(m·s), 50–68° N og 30° V–15° Ø")
    tidsakse(ax); komma(ax)
    plt.show()

**Oppgaver**
1. Når var elven størst? Stemmer det med når det regnet mest?
2. Endre terskelen. Hvorfor bruker forskere ofte både en fast terskel og krav til lengde og bredde for å definere en atmosfærisk elv?

✏️ **Svar**

*Skriv svaret ditt her.*